In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent


C:\Users\91897\AppData\Local\Temp\ipykernel_496\584291432.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
len(docs)

9

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [6]:
len(splitted_docs)

26

In [7]:
# Groq has no embeddings API, so use local Ollama embeddings (no OpenAI key needed).
# Run once in terminal: ollama pull nomic-embed-text
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [8]:
### agent = tools, llm, prompt

In [9]:

@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context += doc.page_content + "\n\n"
    
    return context




In [10]:
# Same pattern as notebooks 5_basic_agents / dynamic.ipynb, but 120b for reliable tool-calling
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [11]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [12]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [13]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  patient name doctor name


In [14]:
result = response["messages"][-1].content

In [15]:
print(result)

**Patient name:** Ms. Nikita Chudhary  

**Doctors mentioned in the report**

- Dr. Nitin Nahar  
- Dr. Swapnil Gupta, MD (Pathology, Chief of Laboratory, Dr Lal PathLabs Ltd)  
- Dr. Kiran Bhargava Pathak, MD (Pathology, Chief of Laboratory, Dr Lal PathLabs Ltd)  
- Dr. Sunanda, MD (Senior Consultant Pathologist – Hematology & Immunology, Dr Lal PathLabs Ltd)  
- Dr. Beena Chandrasekhar, PhD (Technical Director – Flowcytometry, NRL – Dr Lal PathLabs Ltd)  
- Dr. Nagarjun Sai Jaine, MD (Consultant – Hemato‑Oncopathology & Flowcytometry, NRL – Dr Lal PathLabs Ltd)  

These names appear throughout the PDF excerpt, with the patient identified as “Ms. NIKITA CHUDHARY” and the various physicians listed as the reporting and consulting doctors.
